In [1]:
import pandas as pd
import numpy as np
import sys
sys.path.insert(0, "../../run")
from run_config import REPO_PATH, DATA_PROCESSOR_PATH, PROCESSED_DATA_PATH, SEASONS, FEATURES_PATH
import sys
sys.path.insert(1, f"{REPO_PATH}")

import pandas as pd
from src.feature.feature_encoders import TeamEncoder, TeamLagFeatureGenerator, PreviousSeasonTeamAverager, TeamRestDaysCalculator, TeamLagTargetFeature


In [2]:

seasons=sorted(SEASONS)
data_dfs=[pd.read_csv(f"{PROCESSED_DATA_PATH}/{season}/all_data_df.csv") for season in seasons]
target_dfs= [pd.read_csv(f"{PROCESSED_DATA_PATH}/{season}/all_target_df.csv") for season in seasons]
print(len(data_dfs), 'seasons loaded')

key_columns = ['home', 'away', 'date']

# Team Encoding
encoder = TeamEncoder(n_first_matches=5)
encoder.fit(data_dfs)  # season_dfs is a list of DataFrames, one per season
encoder.save(f"{DATA_PROCESSOR_PATH}/team_encoder.pkl")

encoder = TeamEncoder.load(f"{DATA_PROCESSOR_PATH}/team_encoder.pkl")
team_encoding_df = encoder.transform(data_dfs).reset_index(drop=True)

# Previous Season Team Average
previous_season_feature= PreviousSeasonTeamAverager(decay_factor=1, date_col='date', home_col='home', away_col='away')
prev_season_feature_df = previous_season_feature.transform(data_dfs).reset_index(drop=True)

# Team Lag Features
generator = TeamLagFeatureGenerator(lookback=5)
team_lag_feature_df = generator.transform(data_dfs[1:]).reset_index(drop=True)

# Team Rest Days
team_rest_days_calculator = TeamRestDaysCalculator()
team_rest_days_features = team_rest_days_calculator.transform(data_dfs[1:]).reset_index(drop=True)

print('Team rest days features shape:', team_rest_days_features.shape)
print('Team lag features shape:', team_lag_feature_df.shape)
print('Team encoding features shape:', team_encoding_df.shape)
print('Previous season features shape:', prev_season_feature_df.shape)

8 seasons loaded


KeyboardInterrupt: 

### Example: TeamLagTargetFeature - Compute lagged average target features

In [4]:
lag_target_feature = TeamLagTargetFeature()
team_lag_target_df = lag_target_feature.transform(data_dfs[1:], target_dfs[1:]).reset_index(drop=True)
print('Team lag target features shape:', team_lag_target_df.shape)
team_lag_target_df.sample(5)

Team lag target features shape: (2660, 103)


,home,away,date,home_lag_1_home_goals,home_lag_1_away_goals,home_lag_1_home_corners,home_lag_1_away_corners,home_lag_1_home_cards,home_lag_1_away_cards,home_lag_1_home_shots,...,away_lag_5_home_goals,away_lag_5_away_goals,away_lag_5_home_corners,away_lag_5_away_corners,away_lag_5_home_cards,away_lag_5_away_cards,away_lag_5_home_shots,away_lag_5_away_shots,away_lag_5_home_sots,away_lag_5_away_sots
290,Southampton,Tottenham,2019-03-09,3.0,2.0,5.0,7.0,2.0,1.0,15.0,...,1.0,0.0,5.0,3.0,0.0,1.0,21.0,8.0,2.0,2.0
1990,Crystal Palace,Tottenham,2023-10-27,4.0,0.0,2.0,7.0,2.0,3.0,10.0,...,2.0,1.0,7.0,2.0,6.0,7.0,28.0,7.0,10.0,5.0
2004,Newcastle Utd,Arsenal,2023-11-04,2.0,2.0,6.0,6.0,1.0,4.0,11.0,...,1.0,2.0,11.0,4.0,3.0,4.0,12.0,13.0,4.0,5.0
471,West Ham,Sheffield Utd,2019-10-26,2.0,0.0,11.0,2.0,2.0,2.0,19.0,...,0.0,1.0,11.0,6.0,1.0,1.0,17.0,11.0,4.0,7.0
2279,Sheffield Utd,Tottenham,2024-05-19,1.0,0.0,2.0,5.0,1.0,2.0,15.0,...,2.0,2.0,8.0,6.0,2.0,1.0,14.0,9.0,1.0,3.0


In [3]:
prev_season_feature_df['date'].iloc[0], team_encoding_df['date'].iloc[0]

(Timestamp('2022-02-26 00:00:00'), Timestamp('2022-04-10 00:00:00'))

In [ ]:
# Combine all features
combined_features = team_encoding_df.merge(prev_season_feature_df, on=key_columns, how='inner')
combined_features = combined_features.merge(team_lag_feature_df, on=key_columns, how='inner')
combined_features = combined_features.merge(team_rest_days_features, on=key_columns, how='inner')
# Add lag target features as well
combined_features = combined_features.merge(team_lag_target_df, on=key_columns, how='inner')
print('Combined features shape:', combined_features.shape)
combined_features.to_csv(f"{FEATURES_PATH}/all_combined_features.csv", index=False)

Combined features shape: (1510, 2629)


In [8]:
team_rest_days_features

,home,away,date,days_since_last_home,days_since_last_away
0,Leicester City,Crystal Palace,2022-04-10,NaN,NaN
1,Manchester City,Liverpool,2022-04-10,NaN,NaN
2,Norwich City,Burnley,2022-04-10,NaN,NaN
3,Brentford,West Ham,2022-04-10,NaN,NaN
4,Watford,Brentford,2022-04-16,NaN,6.0
...,...,...,...,...,...
1505,Arsenal,Leicester City,2024-09-28,6.0,7.0
1506,Newcastle Utd,Manchester City,2024-09-28,7.0,6.0
1507,Ipswich Town,Aston Villa,2024-09-29,8.0,8.0
1508,Manchester Utd,Tottenham,2024-09-29,8.0,8.0


In [6]:
combined_features[['home', 'away', 'date', 'encoded_away_Tottenham']]

,home,away,date,encoded_away_Tottenham
0,Leicester City,Crystal Palace,2022-04-10,0.0
1,Manchester City,Liverpool,2022-04-10,0.0
2,Norwich City,Burnley,2022-04-10,0.0
3,Brentford,West Ham,2022-04-10,0.0
4,Watford,Brentford,2022-04-16,0.0
...,...,...,...,...
1505,Arsenal,Leicester City,2024-09-28,0.0
1506,Newcastle Utd,Manchester City,2024-09-28,0.0
1507,Ipswich Town,Aston Villa,2024-09-29,0.0
1508,Manchester Utd,Tottenham,2024-09-29,1.0
